In [2]:
import numpy as np 
import time

from reachy import Reachy, parts

from scipy.spatial.transform import Rotation as R

import sys
sys.path.append('../')
from behavior.head_controller import Head_Controller 
from behavior.detection import Detection
from behavior.antenna_moves import Antenna_moves
from behavior.manipulate_flyer import Manipulate_flyer
from behavior.idle import Idle, IdleForever

import behavior.flyer_actions as fa

In [5]:
reachy = Reachy(
    head=parts.Head(io='/dev/ttyUSB*'),
    right_arm=parts.RightArm(io='/dev/ttyUSB*', hand='force_gripper'),
    left_arm=parts.LeftArm(io='/dev/ttyUSB*', hand='flyer_hand')
)

In [10]:
Tab = []

# Base position
x0 = 0.5
y0 = -0.3
z0 = 0
x=0.5

y_mod_a = np.arange(-0.4,0.35,0.05)
z_mod_a = np.arange(-0.15,0.25,0.05)

for y_mod in y_mod_a:
    for z_mod in z_mod_a:
        
        q0 = [0,-31,-7,-103,-111,-7,-10,18]

        # Deviation
        dx = x-x0
        dy = y_mod-y0
        dz = z_mod-z0

        dxyz=[dx,dy,dz]

        alpha = np.arccos((x*x0 + y_mod*y0)/(np.sqrt(x**2+y_mod**2)*np.sqrt(x0**2+y0**2)))
        beta = np.arccos((x*x0 + z_mod*z0)/(np.sqrt(x**2+z_mod**2)*np.sqrt(x0**2+z0**2)))


        # New goal position
        new_pos = reachy.right_arm.forward_kinematics(joints_position=q0).copy()

        for i in range(3):
            new_pos[i][3] = reachy.right_arm.forward_kinematics(joints_position=q0)[i][3] + dxyz[i]

        R23 = np.array(R.from_euler('y', alpha).as_matrix())
        R34 = np.array(R.from_euler('x', beta).as_matrix())
        R12 = reachy.right_arm.forward_kinematics(joints_position=q0)[0:3,0:3]

        R13 = np.dot(R12,R23)
        R14 = np.dot(R13,R34)

        for i in range(3):
            for j in range(3):
                new_pos[i][j] = R14[i][j]


        q0 = [-25,-10,24,-100,-140,0,-40,17]
        JA = reachy.right_arm.inverse_kinematics(new_pos, q0=q0)

        Tab.append([np.round(y_mod,2),np.round(z_mod,2),JA])

/home/pi/.local/lib/python3.7/site-packages/ipykernel_launcher.py:24: RuntimeWarning: invalid value encountered in arccos


In [17]:
print(Tab)

[[-0.4, -0.15, array([-10.39101906, -33.66478464,  -4.73569839, -51.96015156,
       -98.44628178,  18.68037509,  45.        ,  17.        ])], [-0.4, -0.1, array([ -11.53648097,  -12.64409911,   14.97460301,  -96.57000535,
       -100.        ,   -0.65798767,  -36.5906746 ,   17.        ])], [-0.4, -0.05, array([ -13.41490982,  -12.12275379,   12.82753507, -103.04497303,
       -100.        ,   -0.38494426,  -38.18762064,   17.        ])], [-0.4, 0.0, array([ -16.83064451,  -11.3621762 ,   11.30400734, -108.12220694,
       -100.        ,   -0.36421908,  -41.01974066,   17.        ])], [-0.4, 0.05, array([ -20.75576685,   -7.39458771,    8.47052815, -111.53899694,
       -100.        ,   -2.32863967,  -40.57069869,   17.        ])], [-0.4, 0.1, array([ -26.40274083,   -2.58050016,    5.71791185, -113.37862697,
       -100.        ,   -4.22980824,  -40.51171065,   17.        ])], [-0.4, 0.15, array([ -33.34748528,   -1.18347829,    5.61015317, -113.50842209,
       -100.        ,   -5.

In [11]:
np.save('hold_inv_kin.npy',Tab)

In [12]:
a = np.load('/home/pi/dev/reachy-flyers-2/reachy_flyers/notebooks/hold_inv_kin.npy')

In [5]:
print(a)

[[-0.4 -0.15
  array([-10.39101906, -33.66478464,  -4.73569839, -51.96015156,
       -98.44628178,  18.68037509,  45.        ,  17.        ])]
 [-0.4 -0.1
  array([ -11.53648097,  -12.64409911,   14.97460301,  -96.57000535,
       -100.        ,   -0.65798767,  -36.5906746 ,   17.        ])]
 [-0.4 -0.05
  array([ -13.41490982,  -12.12275379,   12.82753507, -103.04497303,
       -100.        ,   -0.38494426,  -38.18762064,   17.        ])]
 [-0.4 0.0
  array([ -16.83064451,  -11.3621762 ,   11.30400734, -108.12220694,
       -100.        ,   -0.36421908,  -41.01974066,   17.        ])]
 [-0.4 0.05
  array([ -20.75576685,   -7.39458771,    8.47052815, -111.53899694,
       -100.        ,   -2.32863967,  -40.57069869,   17.        ])]
 [-0.4 0.1
  array([ -26.40274083,   -2.58050016,    5.71791185, -113.37862697,
       -100.        ,   -4.22980824,  -40.51171065,   17.        ])]
 [-0.4 0.15
  array([ -33.34748528,   -1.18347829,    5.61015317, -113.50842209,
       -100.        ,   -5.

In [19]:
JA = np.ndarray.tolist([tab for (y1,z1,tab) in a if np.logical_and(y1==-0.2,z1==0.1)==True][0])

In [23]:
a[0][0]

-0.4

In [7]:
for(y1,z1,tab) in a:
    if np.logical_and(y1==0.0, z1==0.1)==True:
        JA = tab
print(JA)

[ -42.61200553   10.           52.14646973  -80.79930323 -100.
    7.46064181   -2.10344341   17.        ]


In [19]:
y = 0.0879

In [20]:
y = round(y/0.05)*0.05
print(y)

0.1


In [20]:
for m in reachy.right_arm.motors:
    m.compliant = False

time.sleep(0.01)
print(JA)
reachy.goto({
            m.name: j
            for j, m in zip(JA, reachy.right_arm.motors)
        }, duration=1.5, wait=True, interpolation_mode='minjerk')

[-13.34260996191806, -22.118785858528916, 1.647222586451624, -118.25778123073329, -100.0, -5.771703317572967, -23.86761547344367, 17.0]


In [21]:
for m in reachy.right_arm.motors:
    m.compliant = True

In [3]:
np.square(4)

16

In [17]:
a = [20,89,71,80,72,6,390,92,38]

In [21]:
np.argmin(min(a))

0

In [19]:
max(a)

390

In [2]:
truc = [0,1,2]
bidule = [67,872,93]

print(max(bidule))

print(truc[2])

872
2


In [4]:
if max(bidule) < 2*truc[2]:
    print("coucou")
else:
    print("rien")

rien
